# Exploratory Data Analysis — Credit Card Fraud Detection
**CA 4 MLOps Mini Project**  
Author: Kupakwashe T. Mapuranga

This notebook explores the creditcard.csv dataset and justifies preprocessing decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('../data/raw/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Class Distribution (Imbalance Analysis)

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean()
print(f'Legitimate: {class_counts[0]:,} ({(1-fraud_rate)*100:.2f}%)')
print(f'Fraudulent: {class_counts[1]:,} ({fraud_rate*100:.4f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot(kind='bar', ax=axes[0], color=['steelblue','crimson'], rot=0)
axes[0].set_title('Transaction Class Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])
axes[0].set_ylabel('Count')

axes[1].pie(class_counts, labels=['Legitimate','Fraud'],
            colors=['steelblue','crimson'], autopct='%1.3f%%', startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nConclusion: Severe imbalance → SMOTE oversampling required in preprocessing.')

## 2. Feature Distributions — Amount and Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Amount'].hist(bins=50, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Frequency')

df['Time'].hist(bins=50, ax=axes[1], color='teal', alpha=0.7)
axes[1].set_title('Transaction Time Distribution')
axes[1].set_xlabel('Seconds since first transaction')

plt.tight_layout()
plt.savefig('../reports/amount_time_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Conclusion: Amount and Time are NOT normalized → StandardScaler required.')

## 3. Fraud vs Legitimate — Amount Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, color, name in [(0,'steelblue','Legitimate'), (1,'crimson','Fraud')]:
    df[df['Class']==label]['Amount'].hist(
        bins=50, ax=axes[0], alpha=0.6, label=name, color=color
    )
axes[0].set_title('Amount: Fraud vs Legitimate')
axes[0].set_xlabel('Amount')
axes[0].legend()

df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Amount Boxplot by Class')
axes[1].set_xlabel('Class (0=Legit, 1=Fraud)')

plt.tight_layout()
plt.savefig('../reports/fraud_amount_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Heatmap (Top Features)

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr_with_class.head(15).index.tolist()

plt.figure(figsize=(12, 8))
sns.heatmap(
    df[top_features + ['Class']].corr(),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.5
)
plt.title('Correlation Heatmap — Top 15 Features vs Class')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top features correlated with fraud:')
print(corr_with_class.head(10))

## 5. EDA Summary & Modelling Decisions

| Finding | Decision |
|---------|----------|
| 0.172% fraud rate — severe imbalance | SMOTE oversampling on training data only |
| Amount and Time are unscaled | StandardScaler applied to both features |
| V1–V28 are PCA-transformed | No further scaling needed |
| Strong negative correlation: V14, V12, V10 with fraud | XGBoost will exploit these naturally |
| Binary classification task | XGBoost with scale_pos_weight + AUC-ROC evaluation |